> ## ⚠️ 2026-09-21: 로컬 학습이 기본이 되었습니다
>
> 전체 27,735건 학습이 **16초**(GPU 미사용)로 측정되어 Colab이 필요 없었습니다.
> 평소에는 아래를 쓰세요 — 세션 단위 분할·τ 결정·캐시가 모두 들어 있습니다.
>
> ```bash
> cd services/vision && python training/train_svm.py
> ```
>
> 이 노트북은 **대체 경로**로 남겨둡니다(로컬에 파이썬 환경을 못 만드는 경우 등).
> 다만 2026-09-21 회의 결정(7클래스 학습, 손 방향 축)은 `train_svm.py`에만 반영되어
> 있으니, 이 노트북을 쓸 경우 결과를 그대로 신뢰하지 마세요.

---

# SafeSign 경량 분류기(SVM) 학습 — Colab

**대상**: 심기일전 / vision 담당 (이동혁)
**근거 문서**: `document/05_모델카드_v3.md` (§3-5 정규화, §5 SVM 채택 근거, §6 학습 방법, §7 KPI, §8 τ 결정 절차),
`document/04_데이터셋명세서_v2.md` (클래스·수집 규모·subject-wise split), `document/10_PRD_v2.md`

## 이 노트북이 하는 일

1. 저장소를 clone 해서 **추론과 100% 동일한 정규화 코드**(`services/vision/src/cognition/normalize.py`)를 import
2. `datasets/processed/`의 랜드마크 JSON을 읽어 **63차원 특징벡터**로 변환
3. **인물 단위(subject-wise) 분할** — 외부인 데이터를 test로 완전히 분리 (04_데이터셋명세서_v2 §4)
4. `StandardScaler + SVC(RBF) + 확률 보정(Platt scaling)` 을 **GroupKFold 그리드서치**로 학습
5. 05_모델카드_v3 §8-1 절차대로 **τ(신뢰도 임계값)** 결정
6. match_score 퍼센타일 보정값 + 클래스 템플릿(centroid) 계산
7. `svm_classifier.joblib` 하나로 묶어 내보내기 → 로컬 `services/vision/models/`에 넣으면 끝

## 사전 조건

- 데이터 담당(김지훈)이 `04_데이터셋명세서_v2` 형식으로 수집한 **랜드마크 JSON**이 있어야 한다.
  (원본 이미지가 아니라 **MediaPipe로 이미 추출된 21 keypoints**를 쓴다 — 그래서 이 노트북은 mediapipe/opencv가 필요 없다)
- 학습 자체는 CPU로도 수십 초~수 분이면 끝난다. Colab을 쓰는 이유는 로컬 GPU 성능 한계 때문이며, **GPU 런타임이 필수는 아니다**
  (SVM은 GPU를 쓰지 않는다 — 런타임 유형은 CPU로 둬도 무방).

## 0. 데이터 형식 (중요)

`datasets/processed/{클래스명}/{subject_id}_{index}.json` 을 기대한다. 한 파일은 아래 둘 중 **아무 형식이나** 된다.

**(A) 프레임 1개**
```json
{
  "class_name": "정지",
  "subject_id": "member01",
  "handedness": "Right",
  "landmarks": [{"id": 0, "x": 0.0, "y": 0.0, "z": 0.0}, "... 21개"]
}
```

**(B) 프레임 여러 개(시퀀스)** — 정적 수신호이므로 각 프레임을 독립 샘플로 취급한다
```json
{
  "class_name": "정지",
  "subject_id": "member01",
  "frames": [
    {"handedness": "Right", "landmarks": ["... 21개"]},
    {"handedness": "Right", "landmarks": ["... 21개"]}
  ]
}
```

- `class_name`/`subject_id`는 파일 안에 없으면 **폴더명/파일명에서 유추**한다.
- `landmarks`는 `hand_world_landmarks`(실세계 미터 단위) 기준. 이미지 정규화 좌표(`hand_landmarks`)를 쓰면
  카메라 거리에 따라 값이 달라져 정확도가 떨어진다 (05_모델카드_v3 §3-4).
- `handedness`가 없으면 `"Right"`로 간주한다.

In [ ]:
# @title 1. 저장소 clone + 경로 설정
# 추론(서비스)과 학습(여기)이 **같은 정규화 코드**를 쓰도록 저장소를 그대로 가져온다.
# (정규화가 어긋나면 train-serve skew로 정답률이 조용히 무너진다 — 05_모델카드_v3 §3-5)

REPO_URL = "https://github.com/<팀-계정>/<저장소>.git"  # TODO: 팀 저장소 주소로 교체
REPO_DIR = "/content/SafeSign_PhysicalAI"
BRANCH = "main"

import os, sys, subprocess, shutil

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "services", "vision", "src"))
from cognition.normalize import to_feature_vector, NormalizationError, FEATURE_DIM  # noqa: E402

print("정규화 모듈 로드 완료. FEATURE_DIM =", FEATURE_DIM)

> **저장소가 private이거나 clone이 번거로우면**: 왼쪽 파일 탭에 `services/vision/src/cognition/normalize.py`를
> 업로드하고 `sys.path.insert(0, "/content")` 후 `from normalize import ...` 로 대체해도 된다.
> 단, **로컬 저장소의 normalize.py와 내용이 같아야 한다.**

In [ ]:
# @title 2. 데이터 위치 지정
# 방법 A: Google Drive 마운트 (권장 — 데이터가 크고 재사용하므로)
from google.colab import drive
drive.mount('/content/drive')
PROCESSED_DIR = "/content/drive/MyDrive/SafeSign/datasets/processed"  # TODO: 실제 경로로 교체

# 방법 B: zip 업로드
# from google.colab import files
# up = files.upload()                       # processed.zip 업로드
# !unzip -q processed.zip -d /content/data
# PROCESSED_DIR = "/content/data/processed"

import os
assert os.path.isdir(PROCESSED_DIR), f"경로가 없습니다: {PROCESSED_DIR}"
print(sorted(os.listdir(PROCESSED_DIR)))

In [ ]:
# @title 3. 데이터 적재 → 63차원 특징벡터
import json, glob, collections
import numpy as np

# 02_설계문서_v2 §4 확정 7종 + negative (04_데이터셋명세서_v2 §1과 동일 순서)
SIGN_CLASSES = ["정지", "서행", "좌회전_유도", "우회전_유도", "확인_완료", "후진", "주의", "negative"]
NEGATIVE_CLASS = "negative"


def _iter_samples(path):
    """JSON 1개 파일 -> (handedness, landmarks) 샘플들. 단일 프레임/시퀀스 양쪽 지원."""
    with open(path, encoding="utf-8") as f:
        doc = json.load(f)
    if isinstance(doc, list):                      # [{...}, {...}] 형태도 허용
        frames = doc
    elif "frames" in doc:
        frames = doc["frames"]
    else:
        frames = [doc]
    for fr in frames:
        lms = fr.get("landmarks") if isinstance(fr, dict) else None
        if not lms:
            continue
        yield fr.get("handedness", doc.get("handedness", "Right")), lms, doc


X, y, groups, skipped = [], [], [], collections.Counter()

for class_dir in sorted(glob.glob(os.path.join(PROCESSED_DIR, "*"))):
    if not os.path.isdir(class_dir):
        continue
    class_name = os.path.basename(class_dir)
    if class_name not in SIGN_CLASSES:
        print(f"⚠️ 클래스 폴더명이 정의와 다릅니다(무시): {class_name}")
        continue
    for path in sorted(glob.glob(os.path.join(class_dir, "*.json"))):
        # 파일명 규칙: {subject_id}_{index}.json
        stem = os.path.splitext(os.path.basename(path))[0]
        subject_from_name = stem.rsplit("_", 1)[0] if "_" in stem else stem
        for handedness, lms, doc in _iter_samples(path):
            try:
                feat = to_feature_vector(lms, handedness=handedness)
            except NormalizationError as exc:
                skipped[str(exc)[:40]] += 1
                continue
            X.append(feat)
            y.append(doc.get("class_name", class_name))
            groups.append(str(doc.get("subject_id", subject_from_name)))

X = np.asarray(X, dtype=np.float64)
y = np.asarray(y)
groups = np.asarray(groups)

print(f"샘플 {len(X)}개, 특징 {X.shape[1] if len(X) else 0}차원")
print("클래스별 개수:", dict(collections.Counter(y)))
print("촬영자(subject)별 개수:", dict(collections.Counter(groups)))
if skipped:
    print("정규화 실패로 건너뛴 프레임:", dict(skipped))

assert len(X) > 0, "샘플이 하나도 없습니다 — 경로/폴더 구조를 확인하세요."
missing = set(SIGN_CLASSES) - set(y.tolist())
if missing:
    print(f"⚠️ 아직 데이터가 없는 클래스: {sorted(missing)} — 8개 클래스가 모두 있어야 실전 학습입니다.")

In [ ]:
# @title 4. 분할 — 인물 단위(subject-wise)가 가능하면 그걸로, 아니면 대체 전략
# 04_데이터셋명세서_v2 §4: 같은 사람의 손이 train/test에 섞이면 정답률이 부풀려지므로 인물 단위로 가른다.
#
# ⚠️ 단, **공개 데이터셋은 촬영자 정보가 없어 subject_id가 전부 "public" 하나다.**
#    그룹이 1개면 GroupKFold가 성립하지 않으므로 자동으로 일반 교차검증으로 내려간다 —
#    이때 나오는 점수는 **KPI 근거로 쓸 수 없고**, 자체 촬영 데이터로 따로 평가해야 한다.
import numpy as np

TEST_SUBJECT_IDS = []  # 외부인 subject_id를 넣으면 인물 단위 평가가 켜진다 (예: ["outsider01"])

all_subjects = sorted(set(groups.tolist()))
subject_wise = len(all_subjects) >= 2
print("전체 subject:", all_subjects, "→ 인물 단위 분할", "가능" if subject_wise else "불가")

if TEST_SUBJECT_IDS:
    test_mask = np.isin(groups, TEST_SUBJECT_IDS)
elif subject_wise:
    TEST_SUBJECT_IDS = all_subjects[-1:]
    print(f"⚠️ TEST_SUBJECT_IDS 미지정 — 임시로 {TEST_SUBJECT_IDS}를 test로 씁니다(실전에서는 외부인으로 지정).")
    test_mask = np.isin(groups, TEST_SUBJECT_IDS)
else:
    test_mask = np.zeros(len(groups), dtype=bool)   # 떼어낼 인물이 없다 → test 없음
    print()
    print("🔴 공개 데이터만 있어 인물 단위 분할을 할 수 없습니다(subject가 1개).")
    print("   - 아래 교차검증 점수는 같은 데이터 안에서 나눈 값이라 낙관적으로 부풀려집니다.")
    print("   - 01_프로젝트계획서_v4의 KPI는 **자체 촬영(외부인) 데이터로 측정한 값**으로만 판단하세요.")
    print("     (04_데이터셋명세서_v2 §4 · 05_모델카드_v3 §7-2)")

X_tr, y_tr, g_tr = X[~test_mask], y[~test_mask], groups[~test_mask]
X_te, y_te, g_te = X[test_mask], y[test_mask], groups[test_mask]

print(f"\ntrain/val: {len(X_tr)}개 (subject {sorted(set(g_tr))})")
print(f"test     : {len(X_te)}개 (subject {sorted(set(g_te)) if len(X_te) else '없음'})")
assert len(X_tr) > 0, "train 샘플이 없습니다 — TEST_SUBJECT_IDS가 전체를 가져갔는지 확인하세요."

In [ ]:
# @title 5. 학습: StandardScaler + SVC(RBF) + 확률 보정, 그리드서치 (GroupKFold)
# - SVM(RBF) 채택 근거: 05_모델카드_v3 §5-2 (저차원 63d + 소량 데이터에서 과적합 위험이 낮고 튜닝 파라미터가 적음)
# - 확률 보정: τ 판정에는 확률이 필요한데 SVM은 원래 확률을 내지 않는다. Platt scaling(시그모이드)을 씌운다.
#   `SVC(probability=True)`는 scikit-learn 1.9에서 deprecated(1.11 제거 예정)라 권장 API인
#   CalibratedClassifierCV(SVC(), ensemble=False)를 쓴다 — 동작(Platt scaling)은 동일하다.
# - GroupKFold: CV 분할도 인물 단위로 — 랜덤 KFold를 쓰면 같은 사람이 train/val에 섞여 성능이 부풀려진다.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

SIGN_ONLY = [c for c in SIGN_CLASSES if c != NEGATIVE_CLASS]

def macro_f1_signs(y_true, y_pred):
    """Macro F1은 **신호 7종만** 평균한다 (negative 제외) — 05_모델카드_v3 §7-1 확정."""
    labels = [c for c in SIGN_ONLY if c in set(y_true)]
    return f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

scorer = make_scorer(macro_f1_signs)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", CalibratedClassifierCV(
        SVC(kernel="rbf", random_state=42), method="sigmoid", cv=3, ensemble=False,
    )),
])

param_grid = {
    "clf__estimator__C": [0.1, 1, 10, 100],
    "clf__estimator__gamma": ["scale", 0.01, 0.1, 1],
    # 공개 데이터는 클래스 장수가 최대 18배까지 차이난다(정지 308 vs 좌회전 5,629 —
    # 04_데이터셋명세서_v2 §2-3). "balanced"가 그 불균형을 보정하므로 반드시 후보에 넣는다.
    "clf__estimator__class_weight": [None, "balanced"],
}

# CV 분할기: 인물 단위가 가능하면 GroupKFold, 아니면(공개 데이터 단일 subject) StratifiedKFold
if subject_wise:
    n_splits = min(4, len(set(g_tr)))
    cv, fit_groups = GroupKFold(n_splits=n_splits), g_tr
    print(f"GroupKFold n_splits={n_splits} (인물 단위 — train/val subject 수에 맞춤)")
else:
    n_splits = 5
    cv, fit_groups = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42), None
    print(f"StratifiedKFold n_splits={n_splits} (인물 단위 불가 — 점수가 낙관적으로 나옵니다)")

search = GridSearchCV(pipe, param_grid, scoring=scorer, cv=cv, n_jobs=-1, refit=True, verbose=1)
search.fit(X_tr, y_tr, groups=fit_groups)

print("best params:", search.best_params_)
print(f"best CV macro-F1(7종): {search.best_score_:.4f}")
model = search.best_estimator_

# 참고: 확률 보정의 내부 cv(=3)는 인물 단위가 아니다. 보정 곡선에만 영향을 주고 결정 경계에는
# 영향이 없어서 이대로 두지만, τ를 고르는 확률은 아래 7번에서 **GroupKFold로 다시** 뽑아 쓴다.

In [ ]:
# @title 6. 평가: test(외부인) 기준 KPI 확인
# 목표(01_프로젝트계획서_v4 / 05_모델카드_v3 §7): 정답률 ≥92%, 오분류율 ≤3%, 미판정률 ≤5%,
# Macro F1(7종) ≥0.90, 치명 오분류("정지"→다른 클래스) 0건
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict
import numpy as np

if len(X_te) == 0:
    # 공개 데이터만 있는 경우: 떼어낼 test가 없으므로 교차검증 예측으로 대신 본다.
    print("🔴 test 세트가 없습니다(단일 subject). 교차검증 예측으로 대신 보고합니다 —")
    print("   이 수치는 KPI 근거가 아니라 '파이프라인이 도는지' 확인용입니다.\n")
    y_te = y_tr
    y_pred = cross_val_predict(model, X_tr, y_tr, cv=cv, groups=fit_groups, n_jobs=-1)
else:
    y_pred = model.predict(X_te)

print(classification_report(y_te, y_pred, zero_division=0))
print("Macro F1 (신호 7종):", round(macro_f1_signs(y_te, y_pred), 4))

labels_present = [c for c in SIGN_CLASSES if c in set(y_te.tolist()) | set(y_pred.tolist())]
cm = confusion_matrix(y_te, y_pred, labels=labels_present)
print("\n혼동행렬 (행=정답, 열=예측)")
print("        " + "  ".join(f"{c[:6]:>6}" for c in labels_present))
for name, row in zip(labels_present, cm):
    print(f"{name[:6]:>6}  " + "  ".join(f"{v:>6}" for v in row))

# 치명 오분류: "정지"를 다른 클래스로 판정한 건수 (KPI 0건)
critical = int(((y_te == "정지") & (y_pred != "정지")).sum())
print(f"\n치명 오분류(정지→다른 클래스): {critical}건  {'✅' if critical == 0 else '❌ KPI 위반'}")

In [ ]:
# @title 7. τ(신뢰도 임계값) 결정 — 05_모델카드_v3 §8-1 절차
# 절차: ① 검증셋 확률 수집 → ② τ 그리드 → ③ 치명 오분류 0건 최우선 필터 →
#       ④ 정답률≥92%·오분류율≤3%·미판정률≤5% 동시 만족 → ⑤ 그 중 미판정률 최소 선택
from sklearn.model_selection import cross_val_predict
import numpy as np

# train/val에서 교차검증으로 "편향 없는" 확률을 얻는다 (test는 최종 보고용으로 아껴둔다)
proba_cv = cross_val_predict(
    model, X_tr, y_tr, cv=cv, groups=fit_groups, method="predict_proba", n_jobs=-1,
)
classes_ = np.asarray(model.classes_)          # Pipeline이 최종 추정기의 classes_를 그대로 노출한다
pred_cv = classes_[np.argmax(proba_cv, axis=1)]
conf_cv = proba_cv.max(axis=1)

rows = []
for tau in np.arange(0.50, 0.96, 0.05):
    accepted = conf_cv >= tau
    n = len(y_tr)
    correct = int(((pred_cv == y_tr) & accepted).sum())
    wrong = int(((pred_cv != y_tr) & accepted).sum())
    rejected = int((~accepted).sum())
    critical_t = int(((y_tr == "정지") & (pred_cv != "정지") & accepted).sum())
    rows.append({
        "tau": round(float(tau), 2),
        "정답률": correct / n * 100,
        "오분류율": wrong / n * 100,
        "미판정률": rejected / n * 100,
        "치명오분류": critical_t,
    })

print(f"{'τ':>5} {'정답률':>8} {'오분류율':>9} {'미판정률':>9} {'치명':>5}")
for r in rows:
    print(f"{r['tau']:>5.2f} {r['정답률']:>7.1f}% {r['오분류율']:>8.1f}% {r['미판정률']:>8.1f}% {r['치명오분류']:>5}")

ok = [r for r in rows if r["치명오분류"] == 0
      and r["정답률"] >= 92 and r["오분류율"] <= 3 and r["미판정률"] <= 5]
if ok:
    best = min(ok, key=lambda r: r["미판정률"])   # 조건 만족 중 미판정률 최소 (불필요한 재시도 최소화)
    TAU = best["tau"]
    print(f"\n✅ 선택된 τ = {TAU}  {best}")
else:
    fallback = [r for r in rows if r["치명오분류"] == 0] or rows
    best = max(fallback, key=lambda r: r["정답률"] - r["오분류율"])
    TAU = best["tau"]
    print(f"\n⚠️ 모든 KPI를 동시에 만족하는 τ가 없습니다. 차선책으로 τ={TAU} 선택: {best}")
    print("   → 데이터 추가 수집 또는 클래스 재설계를 06_테스트기록_v2 / 08_리스크레지스터에 기록할 것.")

In [ ]:
# @title 8. match_score 보정 + 클래스 템플릿(centroid) 계산
# 05_모델카드_v3 §2: match_score는 분류와 별개로 "얼마나 비슷한가"를 코사인 유사도로 답한다.
# 퍼센타일 매핑을 위해 **같은 클래스 내 유사도 분포의 5·95 퍼센타일**을 구해 번들에 싣는다.
import numpy as np

templates = {}
for c in sorted(set(y_tr.tolist())):
    templates[c] = X_tr[y_tr == c].mean(axis=0)   # 클래스 centroid (seed_templates.py와 동일 정의)

def cos(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 1e-12 else 0.0

sims = [cos(x, templates[c]) for x, c in zip(X_tr, y_tr) if c in templates]
sim_min, sim_max = float(np.percentile(sims, 5)), float(np.percentile(sims, 95))
print(f"같은 클래스 유사도: p5={sim_min:.4f}, p50={np.percentile(sims, 50):.4f}, p95={sim_max:.4f}")
print("→ 이 구간이 match_score 0~100으로 선형 매핑된다 (services/vision/src/cognition/templates.py)")

In [ ]:
# @title 9. 산출물 저장 + 다운로드
import json, joblib, sklearn, datetime, subprocess, numpy as np

try:
    commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
except Exception:
    commit = None

bundle = {
    "format_version": 1,
    "model": model,
    "classes": [str(c) for c in classes_],
    "tau": float(TAU),
    "n_frames": 3,                                  # 05_모델카드_v3 §8-0 기본값 (실측 후 조정)
    "match_score_calibration": {"sim_min": sim_min, "sim_max": sim_max},
    "metadata": {
        "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "sklearn_version": sklearn.__version__,
        "best_params": search.best_params_,
        "cv_macro_f1_signs": float(search.best_score_),
        "n_train": int(len(X_tr)),
        "n_test": int(len(X_te)),
        "train_subjects": sorted(set(g_tr.tolist())),
        "test_subjects": sorted(set(g_te.tolist())),
        "feature_dim": int(X.shape[1]),
        "repo_commit": commit,
    },
}

joblib.dump(bundle, "svm_classifier.joblib")

# 템플릿은 services/data의 seed_templates.py가 DB에 넣을 수 있게 따로도 내보낸다
with open("sign_templates.json", "w", encoding="utf-8") as f:
    json.dump({k: v.tolist() for k, v in templates.items()}, f, ensure_ascii=False)

print("저장 완료:", bundle["metadata"])

from google.colab import files
files.download("svm_classifier.joblib")
files.download("sign_templates.json")

## 10. 로컬에 적용하기

1. 내려받은 **`svm_classifier.joblib`** 을 로컬 저장소의 아래 경로에 넣는다.

   ```
   services/vision/models/svm_classifier.joblib
   ```

2. vision 서비스를 띄우고 모델이 잡혔는지 확인한다.

   ```bash
   docker compose up --build vision
   curl http://localhost:8001/health      # model.loaded == true, tau 값 확인
   ```

3. **`sign_templates.json`** 은 데이터 담당(김지훈)에게 전달한다 —
   `services/data/src/seed_templates.py`가 이 값을 `sign_templates` 테이블에 넣어야
   `match_score`가 0이 아닌 값으로 나온다.

4. 결과 수치(정답률·Macro F1·τ·혼동행렬)는 `document/05_모델카드_v3.md` §7-3 표와
   `document/06_테스트기록_v2.md`에 옮겨 적는다.

> **재학습이 필요한 때**: 데이터 추가 수집, 클래스(수신호) 정의 변경, 정규화 코드 수정,
> 카메라/촬영 환경 변경. 이 중 **정규화 코드 수정**이 가장 위험하다 — 학습 때와 추론 때가 달라지면
> 정확도가 조용히 떨어지므로, normalize.py를 바꿨으면 **반드시 재학습**할 것.